---
# U-Nets 
---

In this exercise, we will implement a U-Net architecture for diffusion models. 

The U-Net is a convolutional neural network architecture that is widely used for image segmentation tasks. 

It consists of an encoder-decoder structure with skip connections that allow for the preservation of spatial information.

You do not need to read the papers to complete the exercise, but it is highly recommended to read them to gain a deeper understanding of the concepts.

Paper: [U-Net: Convolutional Networks for Biomedical Image Segmentation](https://arxiv.org/abs/1505.04597)

---
# Imports
---

Run the code cell below after you have set up everything correctly (either locally or in Colab).

In [1]:
import importlib
import models
import tests

---
## Time-Conditioning
---

A diffusion model must know the current noise level. Therefore, the U-Net receives both the noisy image $x_t$ and a time value $t$.

We first implement a sinusoidal time embedding. Let `dim` be the embedding dimension and define $h = \left\lfloor \frac{\text{dim}}{2} \right\rfloor$.
For $i = 0, \dots, h-1$, define the frequencies
$$
\omega_i = \exp\left(-\frac{i}{h-1}\log(10000)\right).
$$
For a scalar time value $t$, the sinusoidal time embedding is then
$$
\operatorname{emb}(t)
=
\left[
\sin(t \omega_0), \dots, \sin(t \omega_{h-1}),
\cos(t \omega_0), \dots, \cos(t \omega_{h-1})
\right].
$$

## **Task:**

Implement `SinusoidalTimeEmbedding` in `models.py`.

**Hints:**
- If `dim` is odd, append one zero entry so that the final embedding has dimension `dim`.
- The implementation should accept a tensor `t` of shape `(batch_size,)`. The values of `t` should be converted to floating point, but not rounded. This allows the same embedding function to be used for integer DDPM time indices $t \in \{0, \dots, T-1\}$ and continuous score-based time values $t \in \mathbb{R}$.
- You might find `self.register_buffer()` useful for storing non-learnable parameters during `__init__`.

In [2]:
importlib.reload(tests)
importlib.reload(models)
tests.test_time_embedding()

✅ PASS: Time embedding seems to be correct.


True

---
## U-Net Architecture
---

A U-Net is a convolutional neural network with an **encoder-decoder** structure. For a visual example, see Figure 1 in [U-Net: Convolutional Networks for Biomedical Image Segmentation](https://arxiv.org/abs/1505.04597).

The **encoder** gradually 
- reduces the spatial resolution of the image,
- increases the number of feature channels. 

This allows the network to extract high-level features from the noisy image.

The **decoder** then 
- upsamples these features back to the original image resolution. 

To avoid losing fine-grained spatial information, U-Nets use **skip connections**: feature maps from the encoder are concatenated to feature maps in the decoder at the same resolution.

---

### Time 

In diffusion models, the denoising U-Net must be conditioned on the current diffusion time step $t$. 

To provide this information to the network, first convert $t$ into a sinusoidal time embedding. Then pass this embedding through a small MLP:

$$(B, t) \stackrel{\text{SinusoidalTimeEmbedding}}{\rightarrow} (B, \text{time\_dim}) \stackrel{\text{FC, SiLU}}{\rightarrow} (B, 4 \cdot \text{time\_dim}) \stackrel{\text{FC}}{\rightarrow} (B, \text{time\_dim})$$

### ResBlock

The residual convolutional block `ResBlock` refine feature maps while incorporating the time information. 

They consist of two pre-activation convolutional blocks (`ConvBlock`) using 
- GroupNorm (GN),
- SiLU activation,
- $3\times3$ convolution.

After the first block, the time embedding is projected to `out_channels`, and added to the feature map:
$$(B, C_{in}, H, W) \stackrel{\text{ConvBlock\_1: GN, SiLU, Conv}}{\rightarrow} (B, C_{out}, H, W)$$
$$(B, \text{time\_dim}) \stackrel{\text{SiLU, FC}}{\rightarrow} (B, C_{out})$$
$$(B, C_{out}, H, W) + (B, C_{out}) \stackrel{\text{ConvBlock\_2: GN, SiLU, Conv}}{\rightarrow} (B, C_{out}, H, W)$$

Finally, add a residual skip connection from the input. If `C_in != C_out`, use a $1\times1$ convolution to match the number of channels

---

 ### Architecture

The U-Net used here is intentionally compact, but still follows the standard encoder–bottleneck–decoder structure with skip connections.
First, the input image is projected to $H$ feature channels:
$$(B,3,32,32) \stackrel{\text{Conv}}{\rightarrow} (B,C,32,32)$$

Encoder:
$$
\begin{aligned}
(B,C,32,32)
&\stackrel{\text{ResBlock}}{\rightarrow}
\underbrace{(B,C,32,32)}_{\texttt{skip1}}
\stackrel{\text{Downsample}}{\rightarrow}
(B,C,16,16)
\\
&\stackrel{\text{ResBlock}}{\rightarrow}
\underbrace{(B,2C,16,16)}_{\texttt{skip2}}
\stackrel{\text{Downsample}}{\rightarrow}
(B,2C,8,8)
\\
&\stackrel{\text{ResBlock}}{\rightarrow}
\underbrace{(B,4C,8,8)}_{\texttt{skip3}}
\stackrel{\text{Downsample}}{\rightarrow}
(B,4C,4,4)
\end{aligned}
$$
The downsampling operation is implemented as a $4 \times 4$ convolution with stride $2$.

Bottleneck:
$$
(B,4C,4,4)
\stackrel{\text{ResBlock}}{\rightarrow}
(B,4C,4,4)
$$

Decoder with skip connections:
$$
\begin{aligned}
(B,4C,4,4)
&\stackrel{\text{Upsample}}{\rightarrow}
(B,4C,8,8)
\oplus
\underbrace{(B,4C,8,8)}_{\texttt{skip3}}
\rightarrow
(B,8C,8,8)
\stackrel{\text{ResBlock}}{\rightarrow}
(B,4C,8,8)
\\
&\stackrel{\text{Upsample}}{\rightarrow}
(B,4C,16,16)
\oplus
\underbrace{(B,2C,16,16)}_{\texttt{skip2}}
\rightarrow
(B,6C,16,16)
\stackrel{\text{ResBlock}}{\rightarrow}
(B,2C,16,16)
\\
&\stackrel{\text{Upsample}}{\rightarrow}
(B,2C,32,32)
\oplus
\underbrace{(B,C,32,32)}_{\texttt{skip1}}
\rightarrow
(B,3C,32,32)
\stackrel{\text{ResBlock}}{\rightarrow}
(B,C,32,32)
\end{aligned}
$$
Here, $\oplus$ denotes concatenation along the channel dimension. The upsampling operation first doubles the spatial resolution using nearest-neighbor interpolation and then applies a $3\times3$ convolution.

Finally, the output block maps the feature representation back to the original number of image channels:
$$
(B,C,32,32)
\stackrel{\text{GN, SiLU, Conv}}{\rightarrow}
(B,3,32,32)
$$


## **Task:**

Implement `ResBlock`, `Downsample`, `Upsample`, and `UNet` in `models.py`.

**Hints:**
- The convolutional layers should use a kernel size of 3 and padding of 1 to preserve the spatial resolution.
- Use `bias=True` for convolutional layers in the residual blocks, since the convolutions are applied after `GroupNorm --> SiLU` and are therefore not immediately followed by a normalization layer.
- Use `nn.Upsample` for the upsampling operation.
- Use `nn.GroupNorm` for normalization and ensure that the current channel dimension is divisible by the number of groups.
- You might find `self.register_buffer()` useful for storing non-learnable parameters during `__init__`.

In [3]:
importlib.reload(tests)
importlib.reload(models)
tests.test_resblock()
tests.test_downsample()
tests.test_upsample()
tests.test_unet()

✅ PASS: ResBlock seems to be correct.
✅ PASS: Downsample seems to be correct.
✅ PASS: Upsample seems to be correct.
✅ PASS: UNet seems to be correct.


True

---
## Run Example Training
---
Your implementation should of the U-NET should be able to train a simple DDPM model on MNIST images.

In [5]:
importlib.reload(tests)
importlib.reload(models)
tests.simple_mnist_ddpm()

Training DDPM:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 1/2:   0%|          | 0/938 [00:00<?, ?it/s]

Epoch 1/2 - Loss: 0.0409


Epoch 2/2:   0%|          | 0/938 [00:00<?, ?it/s]

Epoch 2/2 - Loss: 0.0225
Training complete. Generating samples...


Generating samples: 0it [00:00, ?it/s]